# Step 3: NMF Topic Modeling (Dual Model Architecture)
Extracts distinct complaint themes from negative reviews and praise themes from positive reviews using scikit-learn NMF on TF-IDF.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

BASE_DIR = Path(r"C:\Users\Prashma\Desktop\Resume Projects\Data Analytics\NLP-powered customer review analytics system")
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

df = pd.read_csv(PROCESSED_DATA_DIR / "reviews_with_sentiment.csv")
print(f"Loaded {len(df):,} reviews.")


### 3.1 Run Negative & Positive Topic Models

In [ ]:
# Separate keyword rules
NEGATIVE_LABEL_RULES = [
    ("Defective / Hardware Failure", ["defective", "broken", "power", "died", "hardware", "burned", "glitched", "charge", "button", "stopped", "fail", "failed"]),
    ("Durability / Build Quality", ["flimsy", "apart", "cracked", "plastic", "shattered", "snapped", "peeling", "broke", "durability", "junk", "cheaply", "wobbly"]),
    ("Shipping / Packaging Damage", ["shipping", "delivery", "box", "package", "crushed", "delayed", "damaged", "transit", "arrived", "battered", "scratched"]),
    ("Sizing & Dimension Inaccuracy", ["size", "sizing", "dimensions", "fit", "small", "tiny", "gigantic", "measurements", "inaccurate", "clumsy"]),
    ("Overpriced / Poor Value", ["overpriced", "price", "waste", "rip", "money", "expensive", "dollar", "cost", "cash", "worthless"]),
    ("Customer Service & Refund Issues", ["service", "support", "customer", "refund", "warranty", "seller", "emails", "unresponsive", "rude", "incompetent", "runaround"]),
    ("Confusing Manual / Setup Difficulty", ["instructions", "manual", "guide", "steps", "assembly", "setup", "confusing", "translated", "scrambled"]),
    ("Excessive Noise / Chemical Odor", ["noise", "smell", "odor", "loud", "buzzing", "stench", "vibration", "chemical", "screeching", "headache", "clattering"])
]

POSITIVE_LABEL_RULES = [
    ("Superior Build & Durability", ["build", "solid", "quality", "sturdy", "craftsmanship", "robust", "materials", "durable", "heavy", "finish", "textures"]),
    ("Great Value & Affordability", ["value", "price", "affordable", "penny", "bargain", "steal", "money", "worth", "budget", "competitors"]),
    ("Fast Shipping & Secure Packaging", ["shipping", "delivery", "fast", "packaged", "speedy", "cushioning", "pristine", "arrived", "secure", "record"]),
    ("Ease of Use & Simple Setup", ["easy", "intuitive", "simple", "setup", "plug", "effortless", "straightforward", "convenient", "controls", "smart"]),
    ("High Performance & Power", ["performance", "power", "powerful", "flawlessly", "speed", "results", "blazing", "effective", "smooth", "powerhouse"]),
    ("Outstanding Customer Support", ["service", "support", "customer", "courteous", "seller", "responsive", "communication", "helpful", "dedicated", "prompt"]),
    ("Sleek & Modern Design", ["design", "sleek", "modern", "looks", "aesthetic", "stylish", "gorgeous", "beautiful", "chic", "minimalist", "clean"])
]

def auto_label(top_words, rules):
    for label, kws in rules:
        for kw in kws:
            if any(kw in w.lower() for w in top_words):
                return label
    return " / ".join(top_words[:3]).title()

def fit_nmf(texts, k, rules):
    vec = TfidfVectorizer(max_df=0.8, min_df=5, stop_words="english", ngram_range=(1,2))
    tfidf = vec.fit_transform(texts)
    nmf = NMF(n_components=k, random_state=42, init="nndsvda", max_iter=400)
    doc_topics = nmf.fit_transform(tfidf)
    feats = vec.get_feature_names_out()
    labels = []
    for topic in nmf.components_:
        top_w = [feats[i] for i in topic.argsort()[:-8:-1]]
        labels.append(auto_label(top_w, rules))
    return [labels[i] for i in doc_topics.argmax(axis=1)]

# Assign themes
neg_mask = df["sentiment_label"] == "negative"
pos_mask = df["sentiment_label"] == "positive"

df["topic_theme"] = "Neutral / General Feedback"
df["topic_polarity"] = df["sentiment_label"]

df.loc[neg_mask, "topic_theme"] = fit_nmf(df.loc[neg_mask, "full_review"].tolist(), 10, NEGATIVE_LABEL_RULES)
df.loc[pos_mask, "topic_theme"] = fit_nmf(df.loc[pos_mask, "full_review"].tolist(), 8, POSITIVE_LABEL_RULES)

df.to_csv(PROCESSED_DATA_DIR / "reviews_with_topics.csv", index=False)
print("Saved reviews_with_topics.csv!")


### 3.2 Visualizing Complaint & Praise Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df[df["sentiment_label"] == "negative"]["topic_theme"].value_counts().plot(kind='barh', ax=axes[0], color='#E5484D', edgecolor='black')
axes[0].set_title('Customer Complaint Themes (Negative Reviews)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Review Count')

df[df["sentiment_label"] == "positive"]["topic_theme"].value_counts().plot(kind='barh', ax=axes[1], color='#3ECF9E', edgecolor='black')
axes[1].set_title('Customer Praise Themes (Positive Reviews)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Review Count')

plt.tight_layout()
plt.show()
